# Day 2 — Solution: Functions & Linearity

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)

## E1 — payoffs and the fee hinge

In [ ]:
x = np.linspace(40, 90, 200)
pnl = 300 * (x - 62) - 8
breakeven = 62 + 8 / 300
print(f"breakeven: ${breakeven:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(x, pnl); ax[0].axhline(0, color="k", lw=0.5)
ax[0].axvline(breakeven, color="red", ls="--")
ax[0].set_title("P&L(x) = 300(x-62) - 8")

g = np.linspace(-1, 1, 400)
fee = 0.02 + 0.20 * np.maximum(g - 0.02, 0)
ax[1].plot(g, fee); ax[1].set_title("2-and-20 fee as a function of gross return")
plt.tight_layout(); plt.show()

Breakeven at $62.0267: slope 300, intercept −8. The fee function is linear
above the 2% hurdle (slope 0.20) and flat below — the $\max(\cdot, 0)$
hinge, which is also the payoff shape of a call option (module 15). **The
intercept −8 is why "commission-free" changed trading microstructure:
intercepts are not always small relative to edges.**

## E2 — the hedge

In [ ]:
import os
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["XLE", "SPY"], start="2018-01-01"); A, B = "XLE", "SPY"
else:
    px = synthetic_prices(n_days=500, n_assets=2, seed=9); px.columns = [A, B] = ["A", "B"]
rets = px.pct_change().dropna()

# dollar-matched hedge: $1 in A vs $1 in B
pnl = rets[A] - rets[B]
print(f"unhedged vol {rets[A].std():.5f} | hedged vol {pnl.std():.5f}")

**(b) Expected reasoning.** The matched-size hedge only removes risk if the
two legs *move together*; what actually matters is the covariance-based
slope — beta. $k$ matched by notional ignores both relative vol and
co-movement; day 11's regression finds the slope that minimizes hedged
variance. The hedge P&L's intercept (here ~0) would be the *alpha* — the
residual drift after hedging.

## E3 — reading the market model

- $\beta_i$: slope of i's return on the market's — *exposure*.
  (i) β=1.4, α=0: 40% more market sensitivity, no independent drift — a
  levered-market proxy; up 1.4% on a 1% market day, on average.
- $\alpha_i$: intercept — *drift independent of the market*.
  (ii) β=0, α=0.001: market-neutral, +0.1%/day drift — a stat-arb profile
  (rare, precious, and usually smaller than backtests claim).
- (iii) β=0, α=0: no exposure, no drift — pure noise; cash with variance.
- $\varepsilon_{i,t}$: the residual — the part of the day the model can't
  explain (its variance is i's *idiosyncratic risk*).

## E4 — where linearity lies

**Exemplar answer.** Short 200 shares at $40: P&L(x) = 8000 − 200x is
linear, but *x* has no ceiling — the "line" runs to −∞, while the long
side floors at −8000. A risk model that treats long and short P&L as
symmetric linear bets (same slope magnitude, "just negative") understates
tail risk by an unbounded amount; squeezed shorts blow up exactly there.
Also correct: margin calls make *your own* position path-dependent, so even
a linear instrument produces nonlinear account value.